# Unsupervised Machine Learning Project

Ka Ho CHAN 9930788

## Set Up

In [1]:
import pandas as pd                                      # tables (dataframes)
from mlxtend.preprocessing import TransactionEncoder    # one hot encoding: baskets > T/F table
from mlxtend.frequent_patterns import apriori           # find the frequent itemsets
from mlxtend.frequent_patterns import association_rules  # find the association rules
from icecream import ic

In [2]:
'''
T1 -> bread, tea represented by 
T1 -> bread
T1 -> tea

T2 -> sugar, bulb represented by 
T2 -> sugar
T2 -> bulb

T3-> tea, sugar
...

Item = {bread, tea, sugar, bulb}

TransactionEncoder (blank means false for this representation)
    bread, tea, sugar, bulb
T1    T     T
T2                T      T
T3          T     T      
'''

'\nT1 -> bread, tea represented by \nT1 -> bread\nT1 -> tea\n\nT2 -> sugar, bulb represented by \nT2 -> sugar\nT2 -> bulb\n\nT3-> tea, sugar\n...\n\nItem = {bread, tea, sugar, bulb}\n\nTransactionEncoder (blank means false for this representation)\n    bread, tea, sugar, bulb\nT1    T     T\nT2                T      T\nT3          T     T      \n'

## Step 1 - Read the transaction file

In [3]:
'''
The file has ONE ROW PER LINE ITEM (not one row per backet).
Several rows share the same transaction_id: those rows together are one receipt. 
'''

df = pd.read_csv("../data/transactions.zip", usecols=["TransactionID", "Item"])

print('STEP 1 - File Loaded.')
print(f'     rows (line items): {len(df)}')
print(f'     transactions:      {len(df["TransactionID"].unique())}')
print(f'     columns:           {list(df.columns)}') # list gives better formatting

STEP 1 - File Loaded.
     rows (line items): 8379837
     transactions:      1000000
     columns:           ['TransactionID', 'Item']


## Step 2 - Organize the data: one list of items per transaction

In [4]:
''' 
2a) Clean the product names first
- remove spaces at the start and the end ("    butter " -> "butter")
- put everything in lower case ("Butter" -> "butter")
without this, "Butter" and "butter" will be treated as different items, and "    butter " will be treated as a different item from "butter"
'''
df['Item'] = df['Item'].str.strip().str.lower()

''' 
2b) Group the rows by receipts (transaction_id), and collect the items into a list for each transaction.
- gather the item values of each transaction
- run a small function on each group
- remove the duplicates (if any) in each basket (transaction)
- (an itme scanned twice is still the same item)
- put them in a fixed order (sorted) so that the same basket is always represented the same way and easier to read
'''
baskets = df.groupby('TransactionID')['Item'].apply(lambda items: sorted(set(items))
            ) # lambda = a one-line function that takes a list of items and returns a sorted list of unique items (a set that is sorted)
print(baskets)

print('\nSTEP 2 - data organized into baskets')
print(f'     number of baskets:         {len(baskets)}')
print(f'     number of unique items:    {len(set(df['Item']))}')
# print(f'     first 3 baskets:           {baskets.head(3)}') # or baskets.iloc[0:3]
print(f'     first 3 baskets: ')
for b in baskets.head(3):
    print(f'         basket: {b}')

TransactionID
100000     [cidre de glace, couches, fromage cheddar, lin...
100001     [biscuits, bière, bretzels, croustilles, froma...
100002     [ail, crevettes, crème glacée, noix mélangées,...
100003     [algues nori, beurre d'arachide, bouillon à fo...
100004     [café moulu, croissants, fromage suisse, oeufs...
                                 ...                        
1099995    [beurre, beurre d'arachide, citron, concombre,...
1099996    [bacon, biscuits, café moulu, croissants, eau ...
1099997    [papier essuie-tout, pâtes alimentaires, sac r...
1099998    [beurre d'arachide, biscuits, bière, charbon d...
1099999    [charbon de bois, lait 2%, pain à hamburger, s...
Name: Item, Length: 1000000, dtype: object

STEP 2 - data organized into baskets
     number of baskets:         1000000
     number of unique items:    98
     first 3 baskets: 
         basket: ['cidre de glace', 'couches', 'fromage cheddar', 'lingettes pour bébé', 'purée pour bébé', 'sac réutilisable', 'thon en

## Step 3 - Encode the baskets as a True/False Table

In [5]:
''' 
Apriori algorithm requires a one-hot encoded table (T/F table) as input (a table with one row per basket, one column per product), not a list of items.
- use TransactionEncoder to convert the list of items into a one-hot encoded table

TransactionEncoder(): the translator object
    - .fit : learn the list of all products (these become the columns of the table)
    - .transform(baskets): apply it —＞ convert the list of items into a one-hot encoded table
    - te.columns_ : the product names it learned in column order
'''

te = TransactionEncoder() 
translated_data = te.fit(baskets).transform(baskets) # We obtained a matrix of True/False
# print(translated_data, translated_data.shape)
basket_table = pd.DataFrame(translated_data, columns=te.columns_) # convert the matrix into a dataframe with the product names as column names
# basket_table.to_csv('true_false_data.csv', index=False) # save the table to a csv file
# print(basket_table)

print('\nSTEP 3 - basket table built.')
print(f'    shape: {basket_table.shape}    (baskets x products)\n')
print(basket_table.iloc[:3, :5]) # print the first 3 rows and first 5 columns of the table




STEP 3 - basket table built.
    shape: (1000000, 98)    (baskets x products)

     ail  algues nori  avocat  bacon  bagels
0  False        False   False  False   False
1  False        False   False  False   False
2   True        False   False  False   False


## Step 4 - Apropri: Find the frequent itemsets

In [6]:
''' 
apriori() keeps only the combinations of products that appear often enough. 
    - basket_table: the true/false table from Step 3
    - min_support=0.02: Keep an itemset only if it appears in at least 2% of the baskets (transactions)
                          Support is a fraction: 0.02 * 2472 baskets = 49.44 baskets, so we keep itemsets that appear in at least 50 baskets
                          Higher value -> fewer, stronger itemssets and a faster run time
    - use_colnames=True: use the product names (column names) instead of column numbers in the output
                         if false, it will not show the product names, only the column numbers (0, 1, 2, ...) which is not useful
    - max_len=3: Keep only itemsets with 3 or fewer products (1, 2, or 3 products). 
                 Longer ones are raise and hard to act on. Set it to none for no limit
'''

frequent_itemsets = apriori(basket_table, 
                         min_support=0.02, 
                         use_colnames=True, 
                         max_len=3) 

# At this point, the result has 2 columns: 'support' (the fractions) and 'itemsets' (the products in the itemset)

# Add a helper column: how many products are in each itemset
frequent_itemsets['size'] = frequent_itemsets['itemsets'].apply(len) # apply(len) = length of each set

print(frequent_itemsets)

print('\nSTEP 4 - frequent itemsets found by Apriori.')
print(f'  total itemsets: {len(frequent_itemsets)}')
by_size = frequent_itemsets['size'].value_counts().sort_index() # count how many itemsets of each size (1, 2, 3) and sort by size
print(f'  by size: {by_size.to_dict()}') # print the counts as a dictionary
print('  the 5 most frequent items of 2 products: ')
pairs = frequent_itemsets[frequent_itemsets['size']==2].nlargest(5, 'support') # get the 5 pairs with the largest support
# same as: .sort_values('support', ascending=False).head(5)
for i, row in pairs.iterrows():
    print(f'        {row["itemsets"]} : {row["support"]:.3f}') # print the itemset and its support

trash = frequent_itemsets
trash['readable'] = list(trash['itemsets'].apply(lambda x: ', '.join(x))) # convert the set of products into a readable string
trash

      support                                           itemsets  size
0    0.084526                                   frozenset({ail})     1
1    0.068465                           frozenset({algues nori})     1
2    0.044275                                frozenset({avocat})     1
3    0.085813                                 frozenset({bacon})     1
4    0.062716                                frozenset({bagels})     1
..        ...                                                ...   ...
532  0.040564  frozenset({sauce bbq, sac réutilisable, steak ...     3
533  0.021150  frozenset({wasabi, sac réutilisable, sauce soya})     3
534  0.038892  frozenset({sauce tomate, sac réutilisable, vin...     3
535  0.046313  frozenset({sac réutilisable, saucisses, steak ...     3
536  0.024893  frozenset({sauce bbq, saucisses, steak de boeuf})     3

[537 rows x 3 columns]

STEP 4 - frequent itemsets found by Apriori.
  total itemsets: 537
  by size: {1: 93, 2: 245, 3: 199}
  the 5 most frequent

,support,itemsets,size,readable
0,0.084526,frozenset({ail}),1,ail
1,0.068465,frozenset({algues nori}),1,algues nori
2,0.044275,frozenset({avocat}),1,avocat
3,0.085813,frozenset({bacon}),1,bacon
4,0.062716,frozenset({bagels}),1,bagels
...,...,...,...,...
532,0.040564,"frozenset({sauce bbq, sac réutilisable, steak ...",3,"sauce bbq, sac réutilisable, steak de boeuf"
533,0.021150,"frozenset({wasabi, sac réutilisable, sauce soya})",3,"wasabi, sac réutilisable, sauce soya"
534,0.038892,"frozenset({sauce tomate, sac réutilisable, vin...",3,"sauce tomate, sac réutilisable, vin rouge"
535,0.046313,"frozenset({sac réutilisable, saucisses, steak ...",3,"sac réutilisable, saucisses, steak de boeuf"


## Step 5 - Generate the rules

In [7]:
'''
association_rules() splits every frequent itemset into IF -> THEN rules, and calculates the metrics (quality measures) for each rule.
    - frequent_itemsets: the frequent itemsets from Step 4
    - metric='lift': the metric to use for filtering the rules (lift = 1 means no association, lift > 1 means positive association)
    - min_threshold=1.2: keep only rules with lift >= 1.2
        - lift = confidence / support(Y)
               = 1 -> useless
               > 1 -> positive association (if X is bought, Y is more likely to be bought)
               < 1 -> negative association (if X is bought, Y is less likely to be bought)


'''

rules = association_rules(frequent_itemsets,
                          metric='lift', # the metric to use for filtering the rules
                          min_threshold=1.2) # keep only rules with lift >= 1.2 (lift = 1 means no association, lift > 1 means positive association)
                          
# Keep only the rules we can actually serve
 
# 1) ONE product on the THE side: we recommend single items, not bundles
'''
rules['support'] in the market basket analysis, "support" is the percentage of all transactions 
that contain the items in the rule (the IF and THEN side together), e.g. 0.02 means 2% of all baskets

len(basksets) is the total number of baskets (transactions) in the dataset, e.g. 2472

rules['support'] * len(baskets) is the number of baskets that contain the items in the rule 
    (the IF and THEN side together), e.g. 0.02 * 2472 = 49.44 baskets, so we keep itemsets that appear in at least 50 baskets

.round().astype(int) cleans up the math to turn it into a whole number

.apply(len) == 1: keeps only rules with one product on the THEN side (the consequents)

rules['confidence'] >= 0.30: at least 30% of time when the IF side is bought, the THEN side is also bought

at least 40 real baskets behind the rule, so it is not a coincidence: 
    we want to be able to actually serve the recommendation, so we need at least 40 baskets that contain the rule (the IF and THEN side together)
    (support is a fraction, so support x number of baskets = number of baskets that contain the rule)
'''
rules['n_baskets'] = (rules['support'] * len(baskets)).round().astype(int) # portion of baskets that contain the rule (support * total number of baskets)

# print(rules) # use this if I want to see all the rules before the filtering
# print(len (baskets))

rules = rules[(rules['consequents'].apply(len) == 1) &
              (rules['confidence'] >= 0.30) & 
              (rules['n_baskets'] >= 40)] # see above for explanation of the filtering criteria
rules

# print(len (baskets))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,n_baskets
1,frozenset({champignons}),frozenset({ail}),0.056148,0.084526,0.025011,0.445448,5.269949,1.0,0.020265,1.650835,0.858445,0.216240,0.394246,0.370672,25011
2,frozenset({parmesan râpé}),frozenset({ail}),0.092065,0.084526,0.046129,0.501048,5.927740,1.0,0.038347,1.834794,0.915596,0.353582,0.454980,0.523393,46129
3,frozenset({ail}),frozenset({parmesan râpé}),0.084526,0.092065,0.046129,0.545737,5.927740,1.0,0.038347,1.998701,0.908056,0.353582,0.499675,0.523393,46129
4,frozenset({pâtes alimentaires}),frozenset({ail}),0.134353,0.084526,0.071027,0.528660,6.254402,1.0,0.059671,1.942278,0.970503,0.480393,0.485141,0.684479,71027
5,frozenset({ail}),frozenset({pâtes alimentaires}),0.084526,0.134353,0.071027,0.840298,6.254402,1.0,0.059671,5.420378,0.917680,0.480393,0.815511,0.684479,71027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118,"frozenset({sac réutilisable, saucisses})",frozenset({steak de boeuf}),0.075762,0.105492,0.046313,0.611296,5.794713,1.0,0.038321,2.301257,0.895255,0.343209,0.565455,0.525157,46313
1119,"frozenset({sac réutilisable, steak de boeuf})",frozenset({saucisses}),0.095092,0.084141,0.046313,0.487034,5.788303,1.0,0.038312,1.785417,0.914168,0.348428,0.439907,0.518727,46313
1122,"frozenset({sauce bbq, saucisses})",frozenset({steak de boeuf}),0.034745,0.105492,0.024893,0.716448,6.791495,1.0,0.021228,3.154657,0.883453,0.215815,0.683008,0.476209,24893
1123,"frozenset({sauce bbq, steak de boeuf})",frozenset({saucisses}),0.045012,0.084141,0.024893,0.553030,6.572661,1.0,0.021106,2.049040,0.887817,0.238759,0.511967,0.424439,24893


In [8]:
# Sort the best rules first, i.e. highest lift first (the higher the lift, the stronger the association)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True) # reset the index to be 0, 1, 2, ... instead of the original index from the dataframe
rules
# ic(rules)

print('\nSTEP 5 - association rules generated.')
print(f'  Usable rules: {len(rules)}')
print('  The 5 best rules (highest lift): ')
print(rules.head(5)[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'n_baskets']]) # print the first 5 rules with the relevant columns since the rules have been sorted

print("\nAnother way to print the rules in a more readable format:")
for _, r in rules.head(5).iterrows():
    print(f'        {r["antecedents"]} -> {r["consequents"]} : support={r["support"]:.3f}, confidence={r["confidence"]:.3f}, lift={r["lift"]:.3f}, n_baskets={r["n_baskets"]}') # print the rule and its metrics
    
print("\nAnother way to print the rules without the word frozenset:")
for _, r in rules.head(5).iterrows():
    left = ', '.join(r["antecedents"]) # frozenset -> readable text
    right = ', '.join(r["consequents"]) 
    print(f"    IF {left:<28} | THEN {right:<16} | support={r['support']:.3f}, confidence={r['confidence']:.3f}, lift={r['lift']:.3f}, n_baskets={r['n_baskets']}") # print the rule and its metrics


STEP 5 - association rules generated.
  Usable rules: 639
  The 5 best rules (highest lift): 
                                      antecedents  \
0  frozenset({sauce à poutine, frites surgelées})   
1            frozenset({algues nori, sauce soya})   
2      frozenset({algues nori, gingembre mariné})   
3                frozenset({wasabi, algues nori})   
4          frozenset({algues nori, saumon frais})   

                      consequents   support  confidence       lift  n_baskets  
0  frozenset({fromage en grains})  0.034102    0.990100  11.862026      34102  
1        frozenset({riz à sushi})  0.035808    0.991088  11.848739      35808  
2        frozenset({riz à sushi})  0.029684    0.988412  11.816754      29684  
3        frozenset({riz à sushi})  0.027347    0.987756  11.808901      27347  
4        frozenset({riz à sushi})  0.025344    0.986493  11.793811      25344  

Another way to print the rules in a more readable format:
        frozenset({'sauce à poutine', 'frites s

## Step 6 - Save the rules to a file

In [9]:
'''
The rules are the "knowledge" of the recommender. We compute them once (slow)
and save them, so the recommendation part can just read the file.

A frozenset cannot be saved to a csv file, so we convert both sides to the text
'''

rules_to_save = rules.copy() # make a copy of the rules dataframe to save to csv
# convert the frozenset to a string. In lambda x is the frozenset, ', '.join(x) converts 
# it to a string with the items separated by commas, which are sorted 
rules_to_save['antecedents'] = rules_to_save['antecedents'].apply(lambda x: ', '.join(sorted(x))) 
rules_to_save['consequents'] = rules_to_save['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules_to_save

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,n_baskets
0,"frites surgelées, sauce à poutine",fromage en grains,0.034443,0.083468,0.034102,0.990100,11.862026,1.0,0.031227,92.575108,0.948362,0.406901,0.989198,0.699332,34102
1,"algues nori, sauce soya",riz à sushi,0.036130,0.083645,0.035808,0.991088,11.848739,1.0,0.032786,102.819584,0.949924,0.426453,0.990274,0.709591,35808
2,"algues nori, gingembre mariné",riz à sushi,0.030032,0.083645,0.029684,0.988412,11.816754,1.0,0.027172,79.080383,0.943716,0.353410,0.987355,0.671647,29684
3,"algues nori, wasabi",riz à sushi,0.027686,0.083645,0.027347,0.987756,11.808901,1.0,0.025031,74.838361,0.941381,0.325622,0.986638,0.657348,27347
4,"algues nori, saumon frais",riz à sushi,0.025691,0.083645,0.025344,0.986493,11.793811,1.0,0.023195,67.844600,0.939342,0.301743,0.985260,0.644744,25344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
634,"couches, lingettes pour bébé",bière,0.055900,0.150468,0.022915,0.409928,2.724356,1.0,0.014504,1.439710,0.670417,0.124909,0.305416,0.281110,22915
635,"couches, sac réutilisable",bière,0.075561,0.150468,0.027675,0.366260,2.434141,1.0,0.016305,1.340506,0.637335,0.139523,0.254013,0.275093,27675
636,couches,bière,0.083966,0.150468,0.030696,0.365577,2.429597,1.0,0.018062,1.339061,0.642344,0.150664,0.253208,0.284790,30696
637,"lingettes pour bébé, sac réutilisable",bière,0.063087,0.150468,0.022404,0.355129,2.360161,1.0,0.012911,1.317367,0.615105,0.117206,0.240910,0.252012,22404


In [10]:
# Keep only useful columns, in a sensible order for reading
rules_to_save = rules_to_save[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'n_baskets']]
rules_to_save.to_csv('association_rules.csv', index=False) # save the rules to a csv file

print('\nSTEP 6 - association rules saved to file.')
rules_to_save


STEP 6 - association rules saved to file.


,antecedents,consequents,support,confidence,lift,n_baskets
0,"frites surgelées, sauce à poutine",fromage en grains,0.034102,0.990100,11.862026,34102
1,"algues nori, sauce soya",riz à sushi,0.035808,0.991088,11.848739,35808
2,"algues nori, gingembre mariné",riz à sushi,0.029684,0.988412,11.816754,29684
3,"algues nori, wasabi",riz à sushi,0.027347,0.987756,11.808901,27347
4,"algues nori, saumon frais",riz à sushi,0.025344,0.986493,11.793811,25344
...,...,...,...,...,...,...
634,"couches, lingettes pour bébé",bière,0.022915,0.409928,2.724356,22915
635,"couches, sac réutilisable",bière,0.027675,0.366260,2.434141,27675
636,couches,bière,0.030696,0.365577,2.429597,30696
637,"lingettes pour bébé, sac réutilisable",bière,0.022404,0.355129,2.360161,22404


## Step 7 - The Recommender

In [11]:
'''
Suggest products for a basket, with an explanation of why we suggest them.

Recommend products based on the customer's existing basket and the association rules learned from historial shopping transactions.

Parameters: 
- basket: a list of products in the customer's basket
- how many: how many suggestions to return (default 3). Maximum number of recommendations of products to return

Returns: 
- a list of dictionaries, each dictionary contains: {'Item', 'score', 'reason'}
    - item: the product to recommend
    - score: list of score rounded to 2 decimal places
    - reason: explanation for the recommendation, i.e. the rule that led to the recommendation, in a readable format
    
Requirements: 
The dataframe of rules or Json file must always exist.

'''

# TODO Load rules from a file, csv or json, so we don't have to recompute them every time we run the program.
# Create a streamlit app that allows the user to input a basket (by keyboard or a list to choose) and get recommendations based on the rules.

def recommend(basket, how_many=3): 
    # Step 7.1 Convert the basket into a set
    '''
    Example: 
    ['pasta', 'ground beef', 'pasta'] -> {'pasta', 'ground beef'}
    Sets remoe duplicates and make membership checks convenient
        
    '''
    basket = set(basket) # convert the basket into a set to remove duplicates and make membership checks convenient  
                         # a set makes the "is it inside?" test easy  
    
    # Step 7.2 Create dictionnaries to store recommendations
    '''
    Dictionarly storing each recommended product and its best life score.add()
    {
        'tomato source': 2.4
        'parmesan': 1.8
    }
    '''
    
    suggestions = {} # product -> best lift score so format
    
    '''
    Dicionary storing the explanation assocaited with each product
    
    The product name is the key, and its explanataion is the value
    
    Example: 
    {
        'tomato sauce': 'because you bought pasta ...'
    }
    '''
    
    reason = {}
    
    # Step 7.3 Examine every association rule
    '''
    'rules' is a panda datafram that contains the association_rules learned in the previous steps
    '.iterrows()' returns an iterator over the rows of the dataframe, each row is a tuple (index, row)
    '_' receives the row index, which we don't need, and 'row' receives the row itself, which is a pandas Series object
    'rule' contains the current row's data
    '''
    
    # Look at every rule we kept in Step 5 (the rules that are strong enough to be useful)
    for _, rule in rules.iterrows(): 
        # Step 7.3.1 Get the antecedent (the IF side)
        '''
        Retrieve the products requried for this rule to apply
        Example: 
        {'pasta', 'ground beef'}
        
        Convert them into a set for easy comparison with the basket
        '''
        
        if_part = set(rule['antecedents'])
        
        # Step 7.3.2 Get the consequent (the THEN side)
        '''
        Retrieve the product that is recommended by this rule
        'list()' converts the consequent collection into a list
        '[0]' selects the first product in the list
        
        '''
        
        # get the first (and only) item in the set of consequents, since we only kept rules with one consequent
        then_part = list(rule['consequents'])[0]
        # print(if_part, then_part)
        
        if if_part.issubset(basket):
            
            # Step 7.4 Check whether the rule applies to the basket
            '''
            '.issubset()' checks whether the antecedent (the IF side) is a subset of the basket
            'not' reverses the result
            If even one required product is missing, skip this rule and move on to the next one
            because the if part must be fully contained in the basket for the rule to apply
            '''
            
            if not if_part.issubset(basket):
                continue

            # Step 7.5 Check whether the recommended product is already in the basket
            '''
            Check whether the recommended product (the THEN side) is already in the basket
            If it is, skip this rule and move on to the next one            
            because we don't want to recommend products that the customer already has
            '''
            
            if then_part in basket:
                continue
            
            # Step 7.6 Keep the strongest rule for each recommended product
            '''
            A product may be recommended by multiple rules, but we want to keep only the strongest one (the one with the highest lift score)
            
            Example: 
            pasta               -> tomato sauce (lift=1.7)
            pasta + ground beef -> tomato sauce (lift=1.8)
            
            In this case, we want to keep the second rule, because it has a higher lift score (1.8 > 1.7)
            
            'or' means the condition is true if the product is New or 
            the current rule has a higher lift score than the previous one (the one already stored in the suggestions dictionary)
            
            
            
            If the product is not already in the suggestions dictionary, add it with its lift score and explanation
            If the product is already in the suggestions dictionary, but this rule has a higher lift score than the previous one, update it with the new lift score and explanation
            Store the lift score for this product in the suggestions dictionary, and store the explanation for this product in the reason dictionary
            
            '''
            if ((then_part not in suggestions) # if the product is not already in the suggestions dictionary, 
                                              # add it with its lift score and explanation
                or (rule['lift'] > suggestions[then_part]) # if the product is already in the suggestions dictionary, 
                ):                                         # but this rule has a higher lift score than the previous one, 
                                                           # update it with the new lift score and explanation
                
                # store the strongest lift score for this product                
                suggestions[then_part] = rule['lift'] 
                
                # Sort the strongest lift score found for this product
                '''
                Store the antecedent products alphabetically and join them into a readable phrase
                
                Example: 
                {'pasta', 'ground beef'} -> 'ground beef, pasta'
               
                '''
                
                trigger = ' and '.join(sorted(if_part)) # sort the antecedent products alphabetically and join them into a readable phrase
                
                # Build an explanation for this product using f-string
                '''
                {trigger} inserts the products that activated the rule (the IF side) into the explanation
                {then_part} inserts the recommended product (the THEN side) into the explanation
                {rule['confidence']:.0%} inserts the confidence score of the rule into the explanation
                {rule['lift']:.2f} inserts the lift score of the rule into the explanation               
                '''
                reason[then_part] = (
                    f"because you bought {trigger} - "
                    f"(confidence={rule['confidence']:.0%} of baskets with"
                    f"{trigger} also contain {then_part}, "
                    f"lift={rule['lift']:.2f})x more often than average "
                    f"(based on {rule['n_baskets']} real baskets)"
                )
                
                # Save the explanation udner the recommended product's name
                # if a stronger rule is found later, it will overwrite the previous explanation with the new one      

    # Step 7.67 Sort the recommendations from the highest score to the lowest score, keep the top ones
    # sorted(..., key = ..., reverse=True) sorts the items in the suggestions dictionary by their lift score in descending order
    best = sorted(suggestions.items(), 
                    key=lambda pair: pair[1], # lambda pair: pair[1] means sort by the second item in the tuple (the lift score)
                    reverse=True)[:how_many] # keep only the top 'how_many' recommendations
    
    '''
    Example of the output format:
    [
        {
            'Item': 'tomato sauce', # the recommended product
            'score': 2.4, # the lift score rounded to 2 decimal places
            'reason': "because you bought tomato sauce ..." # the explanation for the recommendation
        }
    ]
    If no eligible recommendations are found, the output will be an empty list: []
    
    '''
        
    return [
        {
            'Item': item, # the recommended product
            'score': round(score, 2), # the lift score rounded to 2 decimal places
            'reason': reason[item] # the explanation for the recommendation
        }
        for item, score in best # for each recommended product and its lift score in the top recommendations
    ]

## Step 8 - Test the recommend function with a sample basket_table

In [12]:
print("\nSTEP 8 - test the recommend function with a sample basket")

# Note we may need to parse the basket items to match the cleaned product names in the rules (lowercase, no leading/trailing spaces)
test_baskets = [
    ['pasta'],                         
    ['pasta', 'ground beef'], 
    ['beer', 'tortilla chips'],               
    ['diapers'],                 
    ['yogurt', 'berries'],             
    ['shampoo'],
    ['lemon'],
    ['bananas', 'berries']
]

for id, basket in enumerate(test_baskets):
    print(f"\nBasket: {id+1} - {basket}")
    recommendations = recommend(basket, how_many=3)
    if recommendations:
        for rec in recommendations:
            print(f"  Recommend: {rec['Item']} (lift={rec['score']}) - {rec['reason']}")
    else:
        print("  No recommendations found (no rules apply to this basket).")



STEP 8 - test the recommend function with a sample basket

Basket: 1 - ['pasta']
  No recommendations found (no rules apply to this basket).

Basket: 2 - ['pasta', 'ground beef']
  No recommendations found (no rules apply to this basket).

Basket: 3 - ['beer', 'tortilla chips']
  No recommendations found (no rules apply to this basket).

Basket: 4 - ['diapers']
  No recommendations found (no rules apply to this basket).

Basket: 5 - ['yogurt', 'berries']
  No recommendations found (no rules apply to this basket).

Basket: 6 - ['shampoo']
  No recommendations found (no rules apply to this basket).

Basket: 7 - ['lemon']
  No recommendations found (no rules apply to this basket).

Basket: 8 - ['bananas', 'berries']
  No recommendations found (no rules apply to this basket).
